# Working with LLM APIs (Anthropic)

> 📘 **Python Mastery** · Module 17 — LLM Engineering · Lesson 3/7

Everything before this lesson was theory; this is the plumbing you will ship.
One client, five core patterns — basic calls, multi-turn state management,
streaming, thinking modes, and defensive retries — plus the money layer:
token counting, cost estimation, caching, and batching.

> **Note on running this notebook:** the `anthropic` package needs network
> access and an API key, so every **real** SDK call lives in ```python fences
> inside markdown. Every mechanic also has a labelled **OFFLINE SIMULATION**
> code cell below it that you can execute right now — same shapes, canned
> responses — so the patterns land in your fingers either way.

## 🎯 Learning Objectives

- Configure API keys safely with environment variables and `.env` files
- Call `messages.create`, iterate the response's content blocks, and read `usage`
- Build a `ConversationManager` that resends full history because the API is stateless
- Stream responses and enable adaptive thinking with effort levels
- Estimate request costs with `count_tokens` and a reusable cost estimator function
- Handle rate limits with a specific-first except chain and jittered exponential backoff
- Cut costs with prompt caching — and avoid the mistakes that silently invalidate it

## 1. Setup & Key Management

```bash
pip install anthropic
```

The client reads your key from the environment — **never** paste keys into
code, notebooks, or commits:

```bash
# macOS / Linux
export ANTHROPIC_API_KEY="sk-ant-..."

# Windows PowerShell
$env:ANTHROPIC_API_KEY = "sk-ant-..."
```

For local development, keep secrets in a `.env` file that is **git-ignored**,
loaded with `python-dotenv`:

```bash
pip install python-dotenv
```

```python
from dotenv import load_dotenv
load_dotenv()                       # reads .env into os.environ
```

House rules: one key per environment (dev/staging/prod), rotate on schedule,
and treat any key that touched a shared channel as burned.

In [ ]:
import os

# OFFLINE-SIMULATED-safe key check: runs anywhere, makes no network call.
api_key = os.environ.get("ANTHROPIC_API_KEY")     # read from env -- NEVER hardcode

if api_key:
    print(f"key found: {api_key[:12]}...")        # even YOU only ever see the prefix
else:
    print("ANTHROPIC_API_KEY is not set in this process.")
    print("Live cells will fail until you export it (and restart the kernel).")

print("ready for live calls:", bool(api_key))

## 2. Your First Call

The Messages API takes three essentials: `model`, **required** `max_tokens`
(cap on OUTPUT tokens), and `messages` — a list of alternating user/assistant
turns. Optional extras: `system` (standing orders), `temperature`, `tools`.

```python
import anthropic

client = anthropic.Anthropic()                     # reads ANTHROPIC_API_KEY

response = client.messages.create(
    model="claude-opus-5",
    max_tokens=16000,
    system="You are a concise Python tutor.",
    messages=[{"role": "user", "content": "Explain list comprehensions briefly."}],
)

for block in response.content:                     # content is a LIST of blocks
    if block.type == "text":
        print(block.text)

print(response.stop_reason)                        # "end_turn" | "max_tokens" | "tool_use" ...
print(response.usage.input_tokens, response.usage.output_tokens)
```

Reading the response:

| Field | Meaning |
|---|---|
| `response.content` | List of blocks: `{"text", "tool_use", "thinking"}` — always iterate, never assume one |
| `.stop_reason` | Why generation stopped: `end_turn`, `max_tokens`, `tool_use`, … |
| `.usage.input_tokens / output_tokens` | What you were billed |
| `.id`, `.model`, `.role` | Bookkeeping metadata |

Idiom worth memorising — collapse blocks into one string:

```python
text = "".join(b.text for b in response.content if b.type == "text")
```

**OFFLINE SIMULATION** of the parse step:

In [ ]:
# OFFLINE SIMULATION: same response shape, canned body, zero network.
class FakeBlock:
    def __init__(self, type_, text=""):
        self.type, self.text = type_, text

class FakeResponse:
    def __init__(self, blocks, in_tok, out_tok):
        self.content = blocks
        self.stop_reason = "end_turn"
        self.usage = type("U", (), {"input_tokens": in_tok, "output_tokens": out_tok})

def fake_messages_create(model, max_tokens, system, messages):
    """Stand-in for client.messages.create -- identical call signature."""
    question = messages[-1]["content"]
    return FakeResponse(
        [FakeBlock("text", "List comprehensions build a list from a loop in one line. "),
         FakeBlock("text", f"E.g. squares of 1..5: '[n*n for n in range(1, 6)]'. (re: {question!r})")],
        in_tok=23, out_tok=41,
    )

response = fake_messages_create(
    model="claude-opus-5", max_tokens=16000,
    system="You are a concise Python tutor.",
    messages=[{"role": "user", "content": "Explain list comprehensions briefly."}],
)

text = "".join(b.text for b in response.content if b.type == "text")
print(text)
print("-" * 60)
print("stop_reason :", response.stop_reason)
print("billed      :", response.usage.input_tokens, "in /", response.usage.output_tokens, "out")

## 3. Multi-Turn Conversations: the API Is STATELESS

There is no session on the server. Every call must carry the **entire
transcript** again — the model has no memory between requests. Your job:
append the user turn, resend everything, append the assistant's reply.

> 🔍 **Under the Hood:** a "conversation" exists only on your disk. Because the
> full prefix is resent each turn, cost grows super-linearly over a long chat —
> turn 20 re-pays for turns 1–19. This is exactly what prompt caching (section
> 9) attacks: keep the stable prefix byte-identical and the server reuses its
> computed attention state instead of recomputing it.

```python
import anthropic

class ConversationManager:
    """Remembers history so callers don't have to. The API remembers nothing."""

    def __init__(self, client, model="claude-opus-5", system="You are helpful.",
                 max_tokens=2000):
        self.client, self.model = client, model
        self.system, self.max_tokens = system, max_tokens
        self.messages = []                          # full transcript

    def send(self, user_text):
        self.messages.append({"role": "user", "content": user_text})   # 1. append user turn
        response = self.client.messages.create(                        # 2. RESEND everything
            model=self.model, max_tokens=self.max_tokens,
            system=self.system, messages=self.messages,
        )
        reply = "".join(b.text for b in response.content if b.type == "text")
        self.messages.append({"role": "assistant", "content": reply})  # 3. append assistant turn
        return reply
```

Long-running chats: summarise or drop old turns once the transcript gets
expensive — the manager is the single right place to do that.

In [ ]:
# OFFLINE SIMULATION: watch the resent payload grow every turn.
class FakeBlocks:
    text = "ok"

class FakeAnthropic:
    class messages:                                  # namespace mimicking the SDK
        @staticmethod
        def create(model, max_tokens, system, messages):
            turns_seen.append(len(messages))          # record how much we resent
            last_user = [m["content"] for m in messages if m["role"] == "user"][-1]
            return type("R", (), {"content": [type("B", (), {"type": "text",
                             "text": f"Got it: '{last_user}'"})()]})

turns_seen = []

def send(messages, user_text, client, system="You are helpful.", max_tokens=2000):
    messages.append({"role": "user", "content": user_text})
    response = client.messages.create(model="claude-opus-5", max_tokens=max_tokens,
                                      system=system, messages=messages)
    reply = "".join(b.text for b in response.content if b.type == "text")
    messages.append({"role": "assistant", "content": reply})
    return reply

history, sarah = [], FakeAnthropic()
for turn in ["Hi, I'm Sarah.", "Order #4512 arrived cracked.", "So what are my options?"]:
    print(send(history, turn, client=sarah))

print("\nmessages resent per call:", turns_seen, " <- grows every turn; YOU pay for all of it")
print("final transcript roles  :", [m["role"] for m in history])

## 4. Choosing a Model

All tiers share one API — switching models is changing one string.

| Model | Input $/1M tok | Output $/1M tok | Context | Reach for it when |
|---|---|---|---|---|
| `claude-haiku-4-5` | $1 | $5 | 200K | Volume is high and the task is mechanical: tagging, extraction, routing |
| `claude-sonnet-5` | $2 | $10 | 200K | Default production workhorse — balanced quality, speed, price |
| `claude-opus-5` | $5 | $25 | 1M | Hardest reasoning, very long documents, agentic planning |

Routing advice:
- Start new features on **Sonnet**; move down to Haiku when evals prove it suffices, up to Opus when they prove it doesn't.
- Escalate per-request ("cascades"): cheap tier answers, a checker validates, failures go to the expensive tier.
- Keep model IDs in **config**, never scattered literals — tier changes become one-line deploys.

In [ ]:
def pick_model(complexity, daily_volume):
    """complexity: 'mechanical' | 'standard' | 'hard'; daily_volume: requests/day."""
    if complexity == "hard":
        return "claude-opus-5"
    if complexity == "mechanical" or daily_volume > 250_000:
        return "claude-haiku-4-5"
    return "claude-sonnet-5"

workloads = [
    ("ticket autotagging",   "mechanical", 900_000),
    ("docs summariser",      "standard",    12_000),
    ("multi-doc legal Q&A",  "hard",           400),
    ("chat support agent",   "standard",   300_000),
]
for name, cx, vol in workloads:
    print(f"{name:<22} {vol:>9,}/day  ->  {pick_model(cx, vol)}")

## 5. `max_tokens`: Controlling Output Size and Cost

Required parameter; hard ceiling on generated (output) tokens. Guidance:

- **~16k is a sane non-streaming default** — generous headroom without runaway bills.
- If the cap bites, generation stops early and `stop_reason == "max_tokens"`:
  you get a truncated answer, not an error. Always check.
- Need long outputs? **Stream** (next section) so users see progress, or
  continue in a follow-up turn ("continue exactly where you stopped").

```python
if response.stop_reason == "max_tokens":
    ...  # truncated: raise the cap, tighten the prompt, or request a continuation
```

In [ ]:
# OFFLINE SIMULATION: truncation detection logic.
class SimulatedReply:
    def __init__(self, wanted_output_tokens, cap):
        self.stop_reason = "max_tokens" if wanted_output_tokens > cap else "end_turn"

def handle(cap, wanted):
    reply = SimulatedReply(wanted, cap)
    if reply.stop_reason == "max_tokens":
        print(f"cap={cap:>4}, model wanted {wanted:>4}: TRUNCATED -> raise cap / tighten prompt / continue")
    else:
        print(f"cap={cap:>4}, model wanted {wanted:>4}: end_turn -> complete within budget")

handle(256, 700)     # summary job squeezed too hard
handle(16000, 700)   # roomy default

## 6. Streaming: Tokens as They Arrive

Non-streaming calls return only after the FULL answer is generated. Streaming
yields chunks while generation runs — perceived latency drops from "wall of
silence" to instant typing. Use it for anything user-facing or long.

```python
with client.messages.stream(
    model="claude-opus-5",
    max_tokens=8000,
    messages=[{"role": "user", "content": "Draft a 300-word launch announcement."}],
) as stream:
    for text in stream.text_stream:        # chunks arrive as generated
        print(text, end="", flush=True)
    final = stream.get_final_message()     # the complete message afterwards

print("\noutput tokens:", final.usage.output_tokens)
```

**OFFLINE SIMULATION** of consuming a chunk stream:

In [ ]:
# OFFLINE SIMULATION: progressive rendering from a chunk generator.
def fake_text_stream(chunks):
    yield from chunks                       # stand-in for stream.text_stream

launch_chunks = ["Introducing ", "the Atlas X2 ", "-- ", "a battery ", "that lasts ",
                 "weeks, ", "not hours. ", "Shipping ", "this ", "autumn."]

received = []
for chunk in fake_text_stream(launch_chunks):
    received.append(chunk)
    print(chunk, end="", flush=True)        # UI renders immediately

print()
print(f"\n{len(received)} chunks rendered progressively; "
      f"total {sum(len(c) for c in received)} chars "
      f"-- user started reading seconds earlier than with a blocking call.")

## 7. Adaptive Thinking & Effort

Current Claude models can reason internally before answering. You do **not**
hand them a token budget (`budget_tokens` is deprecated/removed); you let them
think adaptively and set how hard they should try:

- `thinking={"type": "adaptive"}` — the model decides how much thinking the problem deserves.
- `output_config={"effort": "low | medium | high | highest | max"}` — the dials: higher effort buys more reasoning at more latency and cost.
- Thinking arrives as separate `thinking` blocks (often summarised for display); the visible answer stays in `text` blocks.

Rule of thumb: effort `low`–`medium` for classification/extraction, `high`+ for
math, multi-constraint analysis, and agentic planning.

```python
response = client.messages.create(
    model="claude-opus-5",
    max_tokens=16000,
    thinking={"type": "adaptive"},               # model manages its own reasoning depth
    output_config={"effort": "high"},            # try harder; costs latency + tokens
    messages=[{"role": "user", "content": knotty_problem}],
)
for block in response.content:
    if block.type == "thinking":
        ...                                      # reasoning trace (may be a summary)
    elif block.type == "text":
        print(block.text)                        # the actual answer
```

For simple tasks, skip thinking entirely — adaptive means it also knows when
*not* to think.

## 8. Token Counting & a Cost Estimator

Before shipping, know what a call costs. The API can count input tokens
exactly, offline of billing:

```python
count = client.messages.count_tokens(
    model="claude-opus-5",
    system=SYSTEM_PROMPT,
    messages=[{"role": "user", "content": user_prompt}],
)
print(count.input_tokens)
```

Pair that with a local estimator for quick math and CI budget checks —
input estimated from characters (~4 chars/token), output taken as given:**

In [ ]:
PRICES_PER_1M = {                      # US$ per 1M tokens: (input, output)
    "claude-haiku-4-5": (1.0, 5.0),
    "claude-sonnet-5":  (2.0, 10.0),
    "claude-opus-5":    (5.0, 25.0),
}
CHARS_PER_TOKEN = 4                    # English-prose heuristic

def cost_estimator(model, in_chars, out_tokens):
    """Estimated USD for ONE request.
    Input side estimated from characters (verify with count_tokens for precision)."""
    p_in, p_out = PRICES_PER_1M[model]
    in_tokens = -(-in_chars // CHARS_PER_TOKEN)          # ceil division, no imports
    return in_tokens / 1e6 * p_in + out_tokens / 1e6 * p_out

jobs = [
    ("summarise 20-page report", 80_000, 1_000),
    ("classify one ticket",         600,    20),
    ("draft a blog post",           900, 3_500),
]
print(f"{'job':<28}{'haiku':>10}{'sonnet':>10}{'opus':>10}")
for name, chars, out in jobs:
    row = "".join(f"{cost_estimator(m, chars, out):>10.5f}" for m in PRICES_PER_1M)
    print(f"{name:<28}{row}")

big = cost_estimator("claude-opus-5", 80_000, 1_000) * 50_000 * 30
print(f"\nsame report job x 1.5M/month on opus: ${big:,.0f} -- now caching (section 10) looks great")

## 9. Error Handling: Specific-First Excepts + Backoff

Two layers protect production calls:

1. The **SDK already retries** transient failures automatically (default
   `max_retries=2` for 429/5xx). Configure it: `anthropic.Anthropic(max_retries=3)`.
2. Around that, write your own loop for bursts — and catch exceptions
   **specific-first**, because the hierarchy nests:

| Exception | Meaning | Action |
|---|---|---|
| `anthropic.RateLimitError` | HTTP 429 | Read `retry-after` header, wait, retry |
| `anthropic.APIStatusError` | Other HTTP errors | Retry only if `status_code >= 500` |
| `anthropic.BadRequestError` | HTTP 400 — malformed request | Never retry; fix the code |
| `anthropic.AuthenticationError` | Bad/absent key | Fix the environment |

```python
import anthropic, random, time

def call_with_retries(make_request, attempts=5):
    for attempt in range(1, attempts + 1):
        try:
            return make_request()                        # closure around messages.create
        except anthropic.RateLimitError as e:            # 429: server tells us how long
            wait = float(e.response.headers.get("retry-after", 2 ** attempt))
        except anthropic.BadRequestError as e:           # 400: OUR bug -- retrying is denial
            raise
        except anthropic.APIStatusError as e:
            if e.status_code < 500:
                raise                                    # 4xx will not heal by waiting
            wait = min(30, 2 ** attempt)                 # 5xx: transient, back off
        time.sleep(wait + random.uniform(0, 0.5))        # jitter avoids thundering herd
    raise RuntimeError("exhausted retries")
```

**OFFLINE SIMULATION** of the backoff loop against a flaky endpoint:

In [ ]:
import random, time

random.seed(42)

class TransientError(Exception):
    """Stands in for RateLimitError / 5xx."""

calls = {"n": 0}
def flaky_endpoint():
    """Fails twice (like a rate-limited burst), then succeeds."""
    calls["n"] += 1
    if calls["n"] <= 2:
        raise TransientError("503 service unavailable")
    return {"status": "ok", "answer": "Refund approved"}

def retry_with_backoff(fn, max_attempts=5, base_delay=0.05, jitter=0.02):
    """Jittered exponential backoff -- the same shape as the SDK's internal loop."""
    for attempt in range(1, max_attempts + 1):
        try:
            result = fn()
            print(f"attempt {attempt}: SUCCESS -> {result}")
            return result
        except TransientError as err:
            delay = base_delay * 2 ** (attempt - 1) + random.uniform(0, jitter)
            print(f"attempt {attempt}: {err} -- sleeping {delay:.3f}s")
            if attempt == max_attempts:
                raise
            time.sleep(delay)

retry_with_backoff(flaky_endpoint)

## 10. Prompt Caching: Pay Once for a Long Prefix

If many requests share a long, byte-identical prefix (system prompt + policy
docs + few-shot examples), mark the prefix as cacheable. The server stores its
computed state and subsequent calls read it at a fraction of the input price;
cache writes cost a one-time premium.

```python
response = client.messages.create(
    model="claude-opus-5",
    max_tokens=2000,
    system=[
        {"type": "text",
         "text": BIG_HANDBOOK,                       # 40k tokens of policy docs
         "cache_control": {"type": "ephemeral"}},    # <-- cacheable breakpoint
    ],
    messages=[{"role": "user", "content": question}],
)
u = response.usage
print(u.cache_creation_input_tokens)   # >0 on first call: wrote the cache
print(u.cache_read_input_tokens)       # >0 later: prefix reused, big savings
print(u.input_tokens)                  # tokens billed outside the cache
```

Prefix matching is literal: the cached region must be **byte-identical from
the start**. Silent invalidators that quietly destroy your savings:

- timestamps, request IDs, or usernames injected into the system prompt
- re-ordering few-shot examples between calls
- "harmless" copy edits to shared instructions
- volatile data placed BEFORE static data (put variables last)

Cache entries expire after a few minutes without hits — steady traffic keeps
them warm.

## 11. Batch Processing: Half Price, Slower

Not every answer is urgent. The **Message Batches API** accepts large sets of
independent requests, processes them asynchronously, and returns results well
within 24 hours — at **50% off** standard input/output pricing. Ideal for
backfills, nightly enrichment, evaluation sweeps, and dataset labeling.

Conceptual flow (check the docs for current endpoints):

```python
# 1. Build N independent request objects (same shape as messages.create bodies).
# 2. Submit them as ONE batch job.
# 3. Poll for completion (or get notified); collect results within 24 h.
#
# Trade-off: latency measured in minutes-to-hours, not seconds.
# Rule: interactive traffic -> sync/stream; everything scheduled -> batch.
```

Combine with caching and tier-routing and typical pipelines get 60–90% cheaper
than naive implementations.

## 12. Rate-Limit Etiquette

Limits are expressed in requests-per-minute and input/output tokens-per-minute,
scaling with your usage tier. Being a good citizen keeps your quotas rising:

- Honor `retry-after` on 429s — the server is literally telling you when.
- Back off **exponentially with jitter**; never hammer on a fixed interval.
- Move scheduled bulk work to the Batch API instead of parallel-looping sync calls.
- Cache long prefixes — fewer input tokens means fewer token-rate-limit units.
- Monitor `usage` fields and alert on sustained >70% of quota, not on the first rejection.

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Hardcoding API keys in notebooks | Keys leak via git history, screenshots, shared files | Environment variables + git-ignored `.env`; rotate anything exposed |
| Assuming one text block | Multi-block responses silently lose content | Always iterate: join all `block.type == "text"` blocks |
| Retrying `BadRequestError` | The same 400 fails identically N times, slower | Catch specific-first; 4xx means fix code, not wait |
| Forgetting to append the assistant reply | Next call violates role alternation → 400 | `ConversationManager` appends BOTH sides, always |
| Timestamps/IDs in the cached prefix | Cache misses silently; input bill doubles | Byte-stable prefix; inject volatile values at the END |
| Treating `max_tokens` errors as rare | Truncated answers ship to users unnoticed | Check `stop_reason`; alert on `"max_tokens"` in prod |

## 💡 Best Practices & Pro Tips

- Create the client **once** at module level; it owns the connection pool and retry config.
- Log `(model, input_tokens, output_tokens, cache_read, stop_reason)` per call — that log becomes your cost dashboard and regression alarm.
- Stream for humans, batch for machines, cache for everyone.
- Pin model IDs in one config module: tier migrations become a one-line change plus an eval run.
- Budget checks belong in CI: assert `cost_estimator(...) < LIMIT` before deploying prompt changes.
- **AI-engineering relevance:** the stateless-resend pattern explains RAG economics (lesson 5) and agent costs (lesson 7) — every loop iteration repurchases the whole context, which is precisely why caching and context hygiene decide profitability.

## 📌 Summary

| API surface | What it does | Key detail |
|---|---|---|
| `anthropic.Anthropic()` | Client reading `ANTHROPIC_API_KEY` | `max_retries` built in (default 2) |
| `client.messages.create(...)` | One completion | `model`, `max_tokens`, `messages`; optional `system` |
| `response.content` | List of blocks | Filter `block.type == "text"` → `block.text` |
| `ConversationManager` | Statefulness on a stateless API | Resend full history every call |
| `client.messages.stream(...)` | Progressive output | `text_stream`, then `get_final_message()` |
| `thinking={"type": "adaptive"}` + `output_config={"effort": ...}` | Managed reasoning depth | `budget_tokens` is gone; effort: low→max |
| `client.messages.count_tokens(...)` | Exact input-token count | Pre-flight cost checks |
| `cache_control={"type": "ephemeral"}` | Prefix reuse at reduced price | Byte-identical prefixes only; verify via `usage.cache_read_input_tokens` |
| Exceptions (`RateLimitError`, `APIStatusError`) | Typed failure handling | Specific-first catches; retry 429/5xx only |

Key takeaways:
- The API is stateless: your transcript, resent every call — manage it centrally.
- Output tokens cost 5× input everywhere; `max_tokens`, caching, and tier routing are your levers.
- Errors are typed and nested: catch specific-first, back off with jitter, never retry your own bugs.
- Measure before optimising: `count_tokens` plus a cost estimator turns pricing folklore into arithmetic.

## 🔗 Next Lesson

Continue with [`../04_Embeddings_And_Vector_DBs/notes.ipynb`](../04_Embeddings_And_Vector_DBs/notes.ipynb) —
turning text into vectors and searching it: similarity metrics, chunking, and
your own working mini vector database.